# Aufgabe 3.2: Fehldiagnose Sortiermaschine (Lösung)

**Modul:** Informatik I - Mission MotoTec  
**Thema:** Operatorrangfolge bei arithmetischen und logischen Ausdrücken

## Briefing

Die automatische Sortiermaschine am Ende der Fertigungsstraße verteilt die fertigen Fahrzeuge auf verschiedene Spuren: Kleinwagen nach links, SUVs nach rechts, Sondermodelle geradeaus. Seit dem Neustart läuft die Sortierung völlig falsch. SUVs landen auf der Kleinwagen-Spur, Kleinwagen werden als Sondermodelle eingestuft. Schichtleiter Özdal hat die Sortierlogik aus dem Backup wiederhergestellt, aber die Bedingungen im Code sehen verdächtig aus. Bevor jemand etwas ändert, muss erst verstanden werden, was der Code tatsächlich tut.

## Stakeholder-Dialog

*Tarek Özdal (Schichtleiter) steht mit verschränkten Armen vor der Sortiermaschine. Hinter ihm stauen sich Fahrzeuge:*

> *"Die Sortiermaschine schickt SUVs in die Kleinwagen-Spur. Seit einer Stunde. Schaut euch die Bedingungen an - aber fasst den Code noch nicht an. Ich will zuerst wissen, was die Maschine gerade tut und warum. Vorhersagen, dann prüfen. Wer einfach drauflos ändert, macht es nur schlimmer."*

## Beispiele

Python wertet Operatoren nicht einfach von links nach rechts aus, sondern nach einer festen **Rangfolge** (Priorität). Operatoren mit höherer Priorität werden zuerst ausgewertet.

**Arithmetische Operatoren** (von hoch nach niedrig):

| Priorität | Operator | Bedeutung |
|-----------|----------|------------|
| hoch | `**` | Potenz |
| | `*`, `/`, `//`, `%` | Multiplikation, Division, Ganzzahldivision, Modulo |
| niedrig | `+`, `-` | Addition, Subtraktion |

**Logische Operatoren** (von hoch nach niedrig):

| Priorität | Operator |
|-----------|----------|
| hoch | `not` |
| | `and` |
| niedrig | `or` |

In [ ]:
# Potenz vor Multiplikation:
print(2 * 3 ** 2)    # Ergebnis: 18  (nicht 36)
print((2 * 3) ** 2)  # Ergebnis: 36  (Klammern ändern die Reihenfolge)

In [ ]:
# and wird vor or ausgewertet:
print(True or True and False)    # Ergebnis: True  (nicht False)
print((True or True) and False)  # Ergebnis: False (Klammern ändern die Reihenfolge)

## Aufgabe

Die folgenden drei Code-Snippets stammen aus der Sortierlogik der Maschine. Für jedes Snippet gilt:

1. Lesen Sie den Code **sorgfältig** durch.
2. Überlegen Sie **zuerst**, was die Maschine ausgibt. Schreiben Sie Ihre Vorhersage in die Markdown-Zelle unter dem Snippet.
3. Führen Sie den Code **erst danach** aus und vergleichen Sie.

---

### Snippet 1: Gewichtssortierung

Die Maschine soll schwere Fahrzeuge (über 2000 kg) auf die SUV-Spur schicken und leichte Fahrzeuge (unter 800 kg), die **nicht** durch 4 teilbar sind, auf eine Sonderspur. Alle anderen sollen geradeaus weiterfahren. Testen Sie mit zwei Fahrzeugen: einem SUV (2100 kg) und einem Kleinwagen (600 kg).

In [ ]:
# Fahrzeug 1: SUV
gewicht = 2100

if gewicht > 2000 or gewicht < 800 and gewicht % 4 != 0:
    print(f"Fahrzeug ({gewicht} kg) -> Kleinwagen-Spur")
else:
    print(f"Fahrzeug ({gewicht} kg) -> Geradeaus")

**Lösung Fahrzeug 1 (2100 kg):**

Ausgabe: `Fahrzeug (2100 kg) -> Kleinwagen-Spur`

**Schritt-für-Schritt-Auswertung:**

`and` hat eine **höhere Priorität** als `or`. Deshalb wertet Python die Bedingung so aus:

```
gewicht > 2000 or (gewicht < 800 and gewicht % 4 != 0)
```

1. `gewicht > 2000` → `2100 > 2000` → `True`
2. Da links von `or` bereits `True` steht, ist die gesamte Bedingung `True` (Short-Circuit-Auswertung). Der rechte Teil wird gar nicht mehr geprüft.
3. Ergebnis: **`True`** → Der SUV landet auf der Kleinwagen-Spur.

**Das Problem:** Der Programmierer wollte vermutlich `(gewicht > 2000 or gewicht < 800) and gewicht % 4 != 0` schreiben, also: "Wenn das Gewicht über 2000 **oder** unter 800 liegt **und** das Gewicht nicht durch 4 teilbar ist". Ohne Klammern greift aber `and` vor `or`.

In [ ]:
# Fahrzeug 2: Kleinwagen
gewicht = 600

if gewicht > 2000 or gewicht < 800 and gewicht % 4 != 0:
    print(f"Fahrzeug ({gewicht} kg) -> Kleinwagen-Spur")
else:
    print(f"Fahrzeug ({gewicht} kg) -> Geradeaus")

**Lösung Fahrzeug 2 (600 kg):**

Ausgabe: `Fahrzeug (600 kg) -> Geradeaus`

**Schritt-für-Schritt-Auswertung:**

Wieder gilt: `and` vor `or`.

```
gewicht > 2000 or (gewicht < 800 and gewicht % 4 != 0)
```

1. `gewicht > 2000` → `600 > 2000` → `False`
2. `gewicht < 800` → `600 < 800` → `True`
3. `gewicht % 4 != 0` → `600 % 4` → `0` → `0 != 0` → `False`
4. `and`-Verknüpfung: `True and False` → `False`
5. `or`-Verknüpfung: `False or False` → `False`
6. Ergebnis: **`False`** → Der Kleinwagen fährt geradeaus statt auf die Kleinwagen-Spur.

**Zusammenfassung Snippet 1:** Der SUV (2100 kg) landet fälschlich auf der Kleinwagen-Spur. Der Kleinwagen (600 kg) fährt geradeaus statt zur Kleinwagen-Spur. Die Sortierung ist in beiden Fällen falsch - alles wegen fehlender Klammern bei `or`/`and`.

---

### Snippet 2: Sortierpunktzahl

Die Maschine berechnet für jedes Fahrzeug eine Sortierpunktzahl. Je nach Punktzahl wird es einer Kategorie zugeordnet. Was ist das Ergebnis der Berechnung?

In [ ]:
punkte = 10 * 2 ** 2 // 10
print(f"Sortierpunktzahl: {punkte}")

**Lösung Sortierpunktzahl:**

Ausgabe: `Sortierpunktzahl: 4`

**Schritt-für-Schritt-Auswertung:**

Rangfolge: `**` (Potenz) vor `*` und `//` (gleiche Stufe, von links nach rechts).

```
10 * (2 ** 2) // 10
```

1. `2 ** 2` → `4` (Potenz hat höchste Priorität)
2. `10 * 4` → `40` (`*` und `//` haben gleiche Priorität, Auswertung von links nach rechts)
3. `40 // 10` → `4` (Ganzzahldivision)
4. Ergebnis: **`4`**

**Typische Fehler:**
- Wer `10 * 2 = 20` zuerst rechnet und dann `20 ** 2 = 400` und `400 // 10 = 40`, erhält fälschlich 40.
- Wer alles strikt von links nach rechts rechnet (`10 * 2 = 20`, `20 ** 2 = 400`, `400 // 10 = 40`), ignoriert die Vorrangigkeit von `**`.
- Das korrekte Ergebnis ist **4**, weil `**` vor `*` und `//` ausgewertet wird.

---

### Snippet 3: Längensortierung

Die Maschine soll Fahrzeuge nach Länge sortieren. Fahrzeuge, die **nicht** 450 cm lang sind, oder deren Länge plus 1 durch 5 teilbar und dann minus 1 gleich 0 ergibt, sollen auf die Kurzfahrzeug-Spur. Was passiert mit einem Fahrzeug der Länge 450 cm?

In [ ]:
laenge = 450

if laenge != 900 / 2 or laenge + 1 % 5 - 1 == 0:
    print(f"Fahrzeug ({laenge} cm) -> Kurzfahrzeug-Spur")
else:
    print(f"Fahrzeug ({laenge} cm) -> Standardfahrzeug-Spur")

**Lösung Längensortierung (450 cm):**

Ausgabe: `Fahrzeug (450 cm) -> Standardfahrzeug-Spur`

**Schritt-für-Schritt-Auswertung:**

Hier spielen arithmetische **und** logische Operatoren zusammen. Arithmetik wird vor Vergleichen ausgewertet, Vergleiche vor logischen Operatoren.

**Linke Seite des `or`:**
```
laenge != 900 / 2
```
1. `900 / 2` → `450.0` (Division ergibt immer `float`)
2. `450 != 450.0` → `False` (Python vergleicht `int` und `float` korrekt)

**Rechte Seite des `or`:**
```
laenge + 1 % 5 - 1 == 0
```
Hier liegt die Falle. `%` hat höhere Priorität als `+` und `-`:
```
laenge + (1 % 5) - 1 == 0
```
3. `1 % 5` → `1` (Modulo hat Vorrang vor Addition/Subtraktion)
4. `450 + 1` → `451`
5. `451 - 1` → `450`
6. `450 == 0` → `False`

**Zusammenführung:**
7. `False or False` → `False`
8. Ergebnis: **`False`** → Das Fahrzeug fährt zur Standardfahrzeug-Spur.

**Das Problem:** Der Programmierer wollte vermutlich `(laenge + 1) % 5 - 1 == 0` schreiben, also prüfen ob `(450 + 1) % 5 - 1` gleich 0 ist. Mit Klammern: `451 % 5 = 1`, dann `1 - 1 = 0`, also `True`. Ohne Klammern wird aber `1 % 5` zuerst berechnet, und die Bedingung ergibt `False`. Auch hier führen fehlende Klammern zu einer falschen Auswertungsreihenfolge.

## Debriefing

*Özdal schaut auf die Analyse und nickt langsam:*

> *"Gut. Jetzt weiß ich, warum die Maschine Unsinn macht. Fehlende Klammern - so einfach und so fatal. Merkt euch das: Wenn ihr `and` und `or` mischt, setzt Klammern. Immer. Dann passiert so ein Mist nicht nochmal. Saubere Arbeit."*